In [1]:
!pip install -q boxmot
!pip install -q "numpy<2.1"
print("RESTART SESSION ab")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 86.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 40.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 22.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompat

In [2]:
import numpy as np, torch, glob
from pathlib import Path
from boxmot import BotSort

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
TRAINED = glob.glob("/kaggle/input/**/osnet_x1_0_cam12.pt", recursive=True)[0]
print("trained:", TRAINED)

t_pre = BotSort(reid_weights=Path("osnet_x0_25_msmt17.pt"), device=DEVICE, half=False)
t_own = BotSort(reid_weights=Path(TRAINED), device=DEVICE, half=False)

def find_model(t):
    for a in dir(t):
        if a.startswith("_"): continue
        v = getattr(t, a, None)
        if hasattr(v, "get_features") or hasattr(v, "extract_features"):
            return a, v
    return None, None

for name, t in [("pretrained", t_pre), ("trained", t_own)]:
    a, m = find_model(t)
    print(f"\n{name}: attr={a} type={type(m)}")
    print("  use_embeddings:", getattr(t, "use_embeddings", "N/A"))
    print("  with_reid:", getattr(t, "with_reid", "N/A"))
    if m is not None:
        sd = None
        for attr in ["model", "net", "backbone"]:
            inner = getattr(m, attr, None)
            if hasattr(inner, "state_dict"):
                sd = inner.state_dict(); break
        if sd:
            k = list(sd)[len(sd)//2]
            print("  sample tensor", k, "->", float(sd[k].flatten()[0]))

trained: /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False


pretrained: attr=None type=<class 'NoneType'>
  use_embeddings: True
  with_reid: N/A

trained: attr=None type=<class 'NoneType'>
  use_embeddings: True
  with_reid: N/A


In [3]:
import numpy as np, torch, glob
from pathlib import Path
from boxmot import BotSort

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
TRAINED = glob.glob("/kaggle/input/**/osnet_x1_0_cam12.pt", recursive=True)[0]
print("trained:", TRAINED)

frame = (np.random.rand(720, 1280, 3) * 255).astype(np.uint8)   # dummy frame
dets = np.array([[400., 100., 600., 400., 0.9, 0.]])

def probe(tag, weights):
    t = BotSort(reid_weights=Path(weights), device=DEVICE, half=False)
    t.update(dets, frame)          # lazy init force
    t.update(dets, frame)
    print(f"\n===== {tag} =====")
    found = []
    for a in dir(t):
        if a.startswith("_"):
            continue
        try:
            v = getattr(t, a)
        except Exception:
            continue
        if isinstance(v, (int, float, str, bool, type(None), list, dict, tuple)):
            continue
        if callable(v) and not hasattr(v, "__dict__"):
            continue
        found.append((a, type(v).__name__))
    print("attrs:", found)

    for a, _ in found:
        v = getattr(t, a)
        for inner in ["", "model", "net", "backbone", "extractor"]:
            obj = v if inner == "" else getattr(v, inner, None)
            if isinstance(obj, torch.nn.Module):
                sd = obj.state_dict()
                if sd:
                    k = sorted(sd)[len(sd) // 2]
                    print(f"  {a}.{inner or '<self>'} -> {k} = {float(sd[k].flatten()[0]):.8f}")
                break

probe("PRETRAINED", "osnet_x0_25_msmt17.pt")
probe("TRAINED", TRAINED)

trained: /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

INFO     BoxMOT v25.0.0 🚀 Python-3.12.13 torch-2.10.0+cu128                                                       
         CUDA:0 (Tesla T4, 14912MiB)

INFO     /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt

INFO     [PID 58] Downloading ReID weights from https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF → 
         /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt

INFO     Downloading osnet_x0_25_msmt17.pt...

... (5 outputs left out)

INFO     /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt

INFO     [PID 58] Found existing ReID weights at                                                                   
         /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt; skipping download.

INFO     Loaded deployed weights from /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt:
         565/565 required tensors, 100.00% required elements


===== TRAINED =====
attrs: [('aabb_detections_for_association', 'method'), ('all_class_tracks', 'method'), ('apply_cmc', 'method'), ('asso_func', 'function'), ('association_distance', 'method'), ('association_similarity', 'method'), ('capabilities', 'TrackerCapabilities'), ('class_catalog', 'ClassCatalog'), ('cmc', 'ECC'), ('cmc_detection_boxes', 'method'), ('configure_class_catalog', 'method'), ('configure_reid', 'method'), ('detection_layout', 'AxisAlignedDetections'), ('filter_outputs_by_geometry', 'method'), ('format_output_row', 'method'), ('format_output_rows', 'method'), ('format_outputs', 'method'), ('geometry_ops', 'BoxGeometryOps'), ('get_active_tracks_for_display', 'method'), ('get_class_dets_n_embs', 'method'), ('get_class_track_state', 'method'), ('get_class_tracks', 'method'), ('get_lost_tracks_for_display', 'method'), ('get_removed_tracks_for_display', 'method'), ('get_track_box_for_display', 'method'), ('get_track_cls_for_display', 'method'), ('get_track_conf_for_displ

In [4]:
import inspect
from boxmot import BotSort

print("=== update ===")
print(inspect.signature(BotSort.update))
print(inspect.getdoc(BotSort.update))

print("\n=== configure_reid ===")
print(inspect.signature(BotSort.configure_reid))
print(inspect.getdoc(BotSort.configure_reid))

print("\n=== requirements / capabilities ===")
t = BotSort(reid_weights=Path(TRAINED), device=DEVICE, half=False)
print("requirements:", t.requirements)
print("capabilities:", t.capabilities)

print("\n=== get_class_dets_n_embs ===")
print(inspect.signature(BotSort.get_class_dets_n_embs))

print("\n=== update source ===")
src = inspect.getsource(BotSort.update)
print(src[:2500])

=== update ===
(self, detections: 'Detections | np.ndarray', frame: 'Frame | np.ndarray | None' = None, *, timestamp_s: 'float | None' = None) -> 'Tracks | np.ndarray'
Advance one sequence with an optional Frame or uint8 HWC BGR image.

The detection representation determines the output representation.
With ``variable_dt=True``, derive elapsed seconds from
``Frame.timestamp_s`` or the capture timestamp argument. Fixed-step
prediction is the default even when timestamp metadata is present.

=== configure_reid ===
(self, spec: 'ReIDEncoderSpec') -> 'None'
Configure a full encoder spec for lazy tracker-owned inference.

Configuration is intentionally separate from :class:`TrackerSpec`,
whose options describe only the tracking algorithm. The encoder is
constructed on the first non-empty update without supplied embeddings.

=== requirements / capabilities ===


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

requirements: TrackerRequirements(embeddings=True, masks=False, frame=True, frame_dimensions_only=False)
capabilities: TrackerCapabilities(family=<TrackerFamily.BOX: 'box'>, geometry_kinds=frozenset({<GeometryKind.AABB: 'aabb'>, <GeometryKind.OBB: 'obb'>}), requires_embeddings=False, accepts_embeddings=True, requires_masks=False, accepts_masks=False, requires_frame=False, accepts_frame=True)

=== get_class_dets_n_embs ===
(self, dets, embs, cls_id)

=== update source ===
    def update(
        self,
        detections: Detections | np.ndarray,
        frame: Frame | np.ndarray | None = None,
        *,
        timestamp_s: float | None = None,
    ) -> Tracks | np.ndarray:
        """Advance one sequence with an optional Frame or uint8 HWC BGR image.

        The detection representation determines the output representation.
        With ``variable_dt=True``, derive elapsed seconds from
        ``Frame.timestamp_s`` or the capture timestamp argument. Fixed-step
        prediction is t

In [5]:
from boxmot import BotSort
from pathlib import Path
import numpy as np

t = BotSort(reid_weights=Path(TRAINED), device=DEVICE, half=False)
print("generates_embeddings :", getattr(t, "generates_embeddings", "MISSING"))
print("_reid_encoder_spec   :", getattr(t, "_reid_encoder_spec", "MISSING"))
print("_reid_device         :", getattr(t, "_reid_device", "MISSING"))

# kya koi flag hai jo isse on karta hai
import inspect
p = inspect.signature(BotSort.__init__).parameters
print("\ninit params:", list(p))
for k in ["with_reid", "use_embeddings", "reid_model", "reid_weights", "reid_preprocess"]:
    if k in p:
        print(f"  {k}: default = {p[k].default}")

# ReIDEncoderSpec kahan se aata hai
import boxmot
for mod in ["boxmot.appearance", "boxmot.appearance.reid", "boxmot.trackers"]:
    try:
        m = __import__(mod, fromlist=["*"])
        names = [n for n in dir(m) if "Spec" in n or "Encoder" in n]
        if names: print(f"\n{mod}:", names)
    except Exception as e:
        pass
        

INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

generates_embeddings : True
_reid_encoder_spec   : None
_reid_device         : cuda:0

init params: ['self', 'track_high_thresh', 'track_low_thresh', 'new_track_thresh', 'track_buffer', 'match_thresh', 'proximity_thresh', 'appearance_thresh', 'use_cmc', 'cmc_method', 'frame_rate', 'fuse_first_associate', 'use_embeddings', 'second_match_thresh', 'unconfirmed_match_thresh', 'unconfirmed_emb_scale', 'removed_stracks_buffer', 'reid_model', 'reid_weights', 'device', 'half', 'reid_preprocess', 'kwargs']
  use_embeddings: default = True
  reid_model: default = None
  reid_weights: default = None
  reid_preprocess: default = None

boxmot.trackers: ['TrackerSpec', 'TrackerSpec']


In [6]:
import numpy as np, torch, glob, cv2
from pathlib import Path
from boxmot import BotSort

VID = glob.glob("/kaggle/input/**/*chunk_010.mp4", recursive=True)[0]
cap = cv2.VideoCapture(VID); cap.set(cv2.CAP_PROP_POS_FRAMES, 12000)
ok, frame = cap.read(); cap.release()
print("frame:", ok, frame.shape)

dets = np.array([[400., 100., 600., 400., 0.9, 0.],
                 [700., 150., 850., 450., 0.8, 0.]])

def embs_from(weights):
    t = BotSort(reid_weights=Path(weights), device=DEVICE, half=False)
    t.update(dets, frame)
    # jo bhi private attr me encoder bana ho use dhoondo
    for a in dir(t):
        if "reid" not in a.lower() and "emb" not in a.lower():
            continue
        v = getattr(t, a, None)
        if v is None or isinstance(v, (bool, str, int, float)):
            continue
        print("   candidate attr:", a, type(v).__name__)
    return t

print("\n--- PRETRAINED ---")
t1 = embs_from("osnet_x0_25_msmt17.pt")
print("\n--- TRAINED ---")
t2 = embs_from(TRAINED)

frame: True (720, 1280, 3)

--- PRETRAINED ---


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

INFO     BoxMOT v25.0.0 🚀 Python-3.12.13 torch-2.10.0+cu128                                                       
         CUDA:0 (Tesla T4, 14912MiB)

INFO     /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt

INFO     [PID 58] Found existing ReID weights at                                                                   
         /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt; skipping download.

INFO     Loaded deployed weights from /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt: 565/565
         required tensors, 100.00% required elements

... (3 outputs left out)

INFO     /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt

INFO     [PID 58] Found existing ReID weights at                                                                   
         /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt; skipping download.

INFO     Loaded deployed weights from /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt:
         565/565 required tensors, 100.00% required elements

   candidate attr: _get_live_reid_encoder method
   candidate attr: _get_live_reid_model method
   candidate attr: _init_live_reid method
   candidate attr: _mark_live_reid_updated method
   candidate attr: _record_embedding_width method
   candidate attr: _reid_model PyTorchBackend
   candidate attr: _reid_weights PosixPath
   candidate attr: _reset_live_reid_sequence method
   candidate attr: _resolve_input_embeddings method
   candidate attr: configure_reid method
   candidate attr: get_class_dets_n_embs method


In [7]:
import numpy as np, torch

def get_emb(t, frame, boxes):
    m = t._reid_model
    for meth in ["get_features", "extract_features", "forward", "__call__"]:
        fn = getattr(m, meth, None)
        if fn is None:
            continue
        try:
            e = fn(boxes, frame)
            return np.asarray(e), meth
        except Exception as ex:
            last = f"{meth}: {type(ex).__name__} {ex}"
    print("   koi method nahi chala |", last)
    return None, None

boxes = np.array([[400., 100., 600., 400.],
                  [700., 150., 850., 450.]])

e1, m1 = get_emb(t1, frame, boxes)
e2, m2 = get_emb(t2, frame, boxes)
print("pretrained:", None if e1 is None else e1.shape, "| method:", m1)
print("trained   :", None if e2 is None else e2.shape, "| method:", m2)

if e1 is not None and e2 is not None:
    a = e1[0] / (np.linalg.norm(e1[0]) + 1e-9)
    b = e2[0] / (np.linalg.norm(e2[0]) + 1e-9)
    print("\ncosine similarity (box 0):", round(float(a @ b), 6))
    print("embeddings BILKUL SAME?" , np.allclose(e1, e2))

# weights khud compare karo
def w_sample(t):
    m = t._reid_model
    for attr in ["model", "net", "backbone", "module"]:
        o = getattr(m, attr, None)
        if isinstance(o, torch.nn.Module):
            sd = o.state_dict()
            k = sorted(sd)[len(sd)//2]
            return k, float(sd[k].flatten()[0])
    return None, None

print("\npretrained weight:", w_sample(t1))
print("trained    weight:", w_sample(t2))

pretrained: (2, 512) | method: get_features
trained   : (2, 512) | method: get_features

cosine similarity (box 0): 0.299222
embeddings BILKUL SAME? False

pretrained weight: ('conv3.0.gate.fc2.weight', -0.037785086780786514)
trained    weight: ('conv3.0.gate.fc2.weight', 0.051826782524585724)


In [8]:
!pip install -q ultralytics
!pip install -q "numpy<2.1"
print("INSTALL DONE — ab session restart karo")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 5.5 MB/s eta 0:00:00
INSTALL DONE — ab session restart karo


In [12]:
import numpy as np, time, cv2, hashlib
from pathlib import Path
from boxmot import BotSort
from ultralytics import YOLO

person_model = YOLO("yolov8n.pt")

def to_dets(b):
    if len(b) == 0:
        return np.empty((0, 6))
    return np.concatenate([b.xyxy.cpu().numpy(),
                           b.conf.cpu().numpy().reshape(-1, 1),
                           b.cls.cpu().numpy().reshape(-1, 1)], axis=1)

def run(weights, tag, prox, app, nframes=6000):
    tr = BotSort(reid_weights=Path(weights), device=DEVICE, half=False,
                 track_buffer=60, proximity_thresh=prox, appearance_thresh=app)
    stream = person_model.predict(source=VID, conf=0.25, classes=[0],
                                  stream=True, verbose=False)
    rows, ids = [], set()
    for i, res in enumerate(stream):
        if i >= nframes:
            break
        out = np.array(tr.update(to_dets(res.boxes), res.orig_img))
        if out.size:
            for r in out:
                ids.add(int(r[4]))
                rows.append("%d,%d,%.1f,%.1f,%.1f,%.1f" % (i, int(r[4]), r[0], r[1], r[2], r[3]))
    h = hashlib.md5("\n".join(rows).encode()).hexdigest()[:12]
    print(f"{tag:35s} | ids {len(ids):4d} | rows {len(rows):6d} | {h}")
    return h

print("prox=0.5, app=0.25 (default):")
run("osnet_x0_25_msmt17.pt", "  pretrained", 0.5, 0.25)
run(TRAINED,                 "  trained",    0.5, 0.25)

print("\nprox=0.9, app=0.5 (appearance ko zyada mauka):")
run("osnet_x0_25_msmt17.pt", "  pretrained", 0.9, 0.5)
run(TRAINED,                 "  trained",    0.9, 0.5)

prox=0.5, app=0.25 (default):


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

INFO     BoxMOT v25.0.0 🚀 Python-3.12.13 torch-2.10.0+cu128                                                       
         CUDA:0 (Tesla T4, 14912MiB)

INFO     /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt

INFO     [PID 58] Found existing ReID weights at                                                                   
         /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt; skipping download.

INFO     Loaded deployed weights from /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt: 565/565
         required tensors, 100.00% required elements

... (16 outputs left out)

INFO     [PID 58] Found existing ReID weights at                                                                   
         /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt; skipping download.

INFO     Loaded deployed weights from /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt:
         565/565 required tensors, 100.00% required elements

  trained                           | ids    2 | rows   5804 | 794783eebf26


'794783eebf26'

In [15]:
# ===== full-video comparison: pretrained vs trained tracks =====
import glob, json
import numpy as np
from pathlib import Path

def find(name):
    h = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return h[0] if h else None

PRE = find("tracks_pre_dev.txt")
OWN = find("tracks_own_dev.txt")
print("pre:", PRE)
print("own:", OWN)

def load(path):
    d = {}
    for line in open(path):
        p = line.strip().split(",")
        f, tid = int(p[0]), int(p[1])
        x, y, w, h = map(float, p[2:6])
        d.setdefault(f, []).append((tid, x, y, w, h))
    return d

pre, own = load(PRE), load(OWN)
frames = sorted(set(pre) | set(own))
print("frames covered:", len(frames))

# 1) kitne frames pe box-count alag hai
diff_count_frames = [f for f in frames if len(pre.get(f, [])) != len(own.get(f, []))]
print("\n1) frames jahan box COUNT alag:", len(diff_count_frames),
      f"({100*len(diff_count_frames)/len(frames):.1f}%)")

# 2) matched boxes ke beech ID mapping (IoU > 0.5)
def iou(a, b):
    ax1, ay1, ax2, ay2 = a[1], a[2], a[1]+a[3], a[2]+a[4]
    bx1, by1, bx2, by2 = b[1], b[2], b[1]+b[3], b[2]+b[4]
    ix = max(0, min(ax2, bx2) - max(ax1, bx1))
    iy = max(0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    union = a[3]*a[4] + b[3]*b[4] - inter
    return inter / union if union > 0 else 0

pair_count = {}
for f in frames:
    P, O = pre.get(f, []), own.get(f, [])
    used = set()
    for pb in P:
        best, bi = 0, -1
        for j, ob in enumerate(O):
            if j in used: continue
            v = iou(pb, ob)
            if v > best: best, bi = v, j
        if best > 0.5 and bi >= 0:
            used.add(bi)
            pair_count[(pb[0], O[bi][0])] = pair_count.get((pb[0], O[bi][0]), 0) + 1

pre_ids = {p for p, o in pair_count}
own_ids = {o for p, o in pair_count}
splits = sum(1 for p in pre_ids if len({o for (pp, o) in pair_count if pp == p and pair_count[(pp,o)] > 20}) > 1)
merges = sum(1 for o in own_ids if len({p for (p, oo) in pair_count if oo == o and pair_count[(p,oo)] > 20}) > 1)
print("\n2) ID mapping (matched boxes, >20 frame overlap):")
print("   pretrained tracks jo trained mein TOOT ke multiple bane:", merges)
print("   trained tracks jo pretrained mein toot ke multiple the :", splits)

# 3) track lengths
def lengths(d):
    L = {}
    for f, boxes in d.items():
        for (t, *_ ) in boxes:
            L[t] = L.get(t, 0) + 1
    return sorted(L.values(), reverse=True)

lp, lo = lengths(pre), lengths(own)
print("\n3) track lengths (frames):")
print("   pretrained:", len(lp), "tracks | top-5:", lp[:5])
print("   trained   :", len(lo), "tracks | top-5:", lo[:5])

# 4) annotated frames ke saath overlap
gt_path = find("tracking_db.video_tracking (2).json") or find("tracking_db.video_tracking.json")
if gt_path:
    db = json.load(open(gt_path))
    key = next(k for k in {e["video"] for e in db} if "chunk_010" in k)
    ann_frames = {e["frame_number"] + 1 for e in db if e["video"] == key and e.get("tracks")}
    d_on_ann = [f for f in diff_count_frames if f in ann_frames]
    print("\n4) annotated frames:", len(ann_frames))
    print("   box-count-diff frames jo annotated hain:", len(d_on_ann),
          f"({100*len(d_on_ann)/max(len(diff_count_frames),1):.1f}% of diffs)")
else:
    print("\n4) GT json attached nahi — tracking_db_json_file dataset attach karke dobara chalao")

pre: /kaggle/input/notebooks/payal1238gupta/phase4-comparison/tracks_pre_dev.txt
own: /kaggle/input/notebooks/payal1238gupta/phase4-comparison/tracks_own_dev.txt
frames covered: 27398

1) frames jahan box COUNT alag: 522 (1.9%)

2) ID mapping (matched boxes, >20 frame overlap):
   pretrained tracks jo trained mein TOOT ke multiple bane: 0
   trained tracks jo pretrained mein toot ke multiple the : 1

3) track lengths (frames):
   pretrained: 12 tracks | top-5: [10704, 10456, 5469, 729, 71]
   trained   : 13 tracks | top-5: [10704, 10456, 4896, 729, 71]

4) annotated frames: 916
   box-count-diff frames jo annotated hain: 13 (2.5% of diffs)


In [16]:
try:
    import ultralytics, boxmot, motmetrics
    print("sab installed")
except ImportError:
    !pip install -q ultralytics boxmot
    !pip install -q --no-deps motmetrics xmltodict
    !pip install -q "numpy<2.1"
    print("INSTALL DONE — Run > Restart Session karo, phir Cell B se")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.3 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 5.7 M

In [17]:
# ===== track_buffer experiment =====
import numpy as np, torch, glob, time
from pathlib import Path
from boxmot import BotSort
from ultralytics import YOLO

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
TRAINED = glob.glob("/kaggle/input/**/osnet_x1_0_cam12.pt", recursive=True)[0]
VID = glob.glob("/kaggle/input/**/*chunk_010.mp4", recursive=True)[0]
person_model = YOLO("yolov8n.pt")
print("device:", DEVICE, "| video:", Path(VID).name)

def to_dets(b):
    if len(b) == 0:
        return np.empty((0, 6))
    return np.concatenate([b.xyxy.cpu().numpy(),
                           b.conf.cpu().numpy().reshape(-1, 1),
                           b.cls.cpu().numpy().reshape(-1, 1)], axis=1)

def run(weights, out_name, buffer):
    tr = BotSort(reid_weights=Path(weights), device=DEVICE, half=False,
                 track_buffer=buffer, proximity_thresh=0.9, appearance_thresh=0.5)
    stream = person_model.predict(source=VID, conf=0.25, classes=[0], stream=True, verbose=False)
    rows, ids, n, t0 = [], set(), 0, time.time()
    for res in stream:
        out = np.array(tr.update(to_dets(res.boxes), res.orig_img))
        if out.size:
            for r in out:
                ids.add(int(r[4]))
                rows.append("%d,%d,%.1f,%.1f,%.1f,%.1f,1,-1,-1,-1" % (n + 1, int(r[4]), r[0], r[1], r[2] - r[0], r[3] - r[1]))
        n += 1
    open(out_name, "w").write("\n".join(rows) + "\n")
    print("%-22s | buffer %3d | unique ids %3d | %.1f min" % (out_name, buffer, len(ids), (time.time() - t0) / 60))

run("osnet_x0_25_msmt17.pt", "pre_buf300.txt", 300)
run(TRAINED,                 "own_buf300.txt", 300)

device: cuda:0 | video: Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_010.mp4


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

INFO     BoxMOT v25.0.0 🚀 Python-3.12.13 torch-2.10.0+cu128                                                       
         CUDA:0 (Tesla T4, 14912MiB)

INFO     /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt

INFO     [PID 58] Found existing ReID weights at                                                                   
         /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt; skipping download.

INFO     Loaded deployed weights from /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt: 565/565
         required tensors, 100.00% required elements

... (3 outputs left out)

INFO     /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt

INFO     [PID 58] Found existing ReID weights at                                                                   
         /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt; skipping download.

INFO     Loaded deployed weights from /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt:
         565/565 required tensors, 100.00% required elements

own_buf300.txt         | buffer 300 | unique ids  10 | 14.8 min


In [18]:
# ===== eval =====
import numpy as np, json, glob, pandas as pd
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)
import motmetrics as mm

GT = glob.glob("/kaggle/input/**/tracking_db.video_tracking*.json", recursive=True)[0]
db = json.load(open(GT))
KEY = next(k for k in {e["video"] for e in db} if "chunk_010" in k)
gt = {}
for e in db:
    if e["video"] != KEY: continue
    rows = [(t["track_id"], t["bbox"][0], t["bbox"][1], t["bbox"][2]-t["bbox"][0], t["bbox"][3]-t["bbox"][1])
            for t in e.get("tracks", [])]
    if rows: gt[e["frame_number"] + 1] = rows
print("GT frames:", len(gt))

POSTER = [385, 84, 622, 381]
def in_poster(x, y, w, h, thr=0.95):
    ix1, iy1 = max(x, POSTER[0]), max(y, POSTER[1])
    ix2, iy2 = min(x+w, POSTER[2]), min(y+h, POSTER[3])
    return max(0, ix2-ix1) * max(0, iy2-iy1) / max(w*h, 1) > thr

def load(path):
    d = {}
    for line in open(path):
        p = line.strip().split(",")
        if len(p) < 6: continue
        f, tid = int(p[0]), int(p[1]); x, y, w, h = map(float, p[2:6])
        if not in_poster(x, y, w, h):
            d.setdefault(f, []).append((tid, x, y, w, h))
    return d

def uids(d): return len({t for v in d.values() for (t, *_) in v})

def evaluate(pred, name):
    acc = mm.MOTAccumulator(auto_id=False)
    for f in sorted(gt):
        g, p = gt[f], pred.get(f, [])
        gb = np.array([r[1:] for r in g], dtype=float).reshape(-1, 4)
        pb = np.array([r[1:] for r in p], dtype=float).reshape(-1, 4)
        acc.update([r[0] for r in g], [r[0] for r in p], mm.distances.iou_matrix(gb, pb, max_iou=0.5), frameid=f)
    return mm.metrics.create().compute(acc, name=name, metrics=["mota", "idf1", "num_switches", "num_false_positives", "num_misses"])

def find(n): return glob.glob(f"/kaggle/input/**/{n}", recursive=True)[0]
files = [("pretrained | buf 60",  find("tracks_pre_dev.txt")),
         ("TRAINED    | buf 60",  find("tracks_own_dev.txt")),
         ("pretrained | buf 300", "pre_buf300.txt"),
         ("TRAINED    | buf 300", "own_buf300.txt")]

res = []
for name, path in files:
    p = load(path)
    r = evaluate(p, name); r["unique_ids"] = uids(p); res.append(r)
print(pd.concat(res).to_string())

GT frames: 916
                          mota      idf1  num_switches  num_false_positives  num_misses  unique_ids
pretrained | buf 60   0.662605  0.622715             7                   12         341          12
TRAINED    | buf 60   0.662605  0.622715             7                   12         341          13
pretrained | buf 300  0.655108  0.605117             6                   13         349          11
TRAINED    | buf 300  0.660731  0.606466             4                    9         349          10
